# Colab Demo: Baseline RAG with Long-T5

This notebook runs baseline RAG with:
- local Colab repo and local virtual environment (fast dependency install)
- Drive-hosted artifacts via a local data symlink
- fixed generator model `google/long-t5-tglobal-base`

Outputs:
- On-screen qualitative results (query, evidence, response, claims)
- JSON file under local `outputs/rag_demo_results_<timestamp>.json`
- Optional copy of output JSON to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

# Local repo for fast environment setup
REPO_DIR = Path('/content/AIST-FYP')
REPO_URL = 'https://github.com/xiashuidaolaoshuren/AIST-FYP.git'

# Drive-hosted artifacts root (same structure as repo data/)
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/data')

if REPO_DIR.exists() and (REPO_DIR / '.git').exists():
    print(f'Using existing local repo: {REPO_DIR}')
    os.system(f'git -C {REPO_DIR} fetch --all')
    os.system(f'git -C {REPO_DIR} checkout main')
    os.system(f'git -C {REPO_DIR} pull --ff-only origin main')
else:
    print('Cloning repo to local Colab storage...')
    clone_code = os.system(f'git clone {REPO_URL} {REPO_DIR}')
    if clone_code != 0:
        raise RuntimeError('Failed to clone repo to local storage.')

if not DRIVE_DATA_ROOT.exists():
    raise FileNotFoundError(
        f'Drive data root not found: {DRIVE_DATA_ROOT}. Create this folder and place artifacts under indexes/ and processed/.'
    )

print('Local repo:', REPO_DIR)
print('Drive data root:', DRIVE_DATA_ROOT)

%cd {REPO_DIR}

In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

print('Installing dependencies...')

# Ensure we start with system Python, not any pre-existing venv
os.environ.pop('VIRTUAL_ENV', None)
original_path = os.environ.get('PATH', '')

repo_dir = Path(REPO_DIR)
uv_project = repo_dir / 'colab' / 'env'
uv_extra = 'evaluation'

# Force uv venv to stay in local repo, never under Drive
os.environ['UV_PROJECT_ENVIRONMENT'] = str(uv_project / '.venv')


def shell_join(parts):
    return ' '.join(shlex.quote(str(p)) for p in parts)


def run(cmd, cwd=None, check=True, stream=True):
    print(f"\n$ {cmd}")
    if stream:
        process = subprocess.Popen(
            cmd,
            shell=True,
            cwd=str(cwd) if cwd is not None else None,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1,
        )
        out_lines = []
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
            out_lines.append(line)
        process.wait()
        completed = subprocess.CompletedProcess(
            args=cmd,
            returncode=process.returncode,
            stdout=''.join(out_lines),
            stderr=''
        )
    else:
        completed = subprocess.run(
            cmd,
            shell=True,
            cwd=str(cwd) if cwd is not None else None,
            text=True,
            capture_output=True,
            check=False
        )
        if completed.stdout:
            print(completed.stdout)
        if completed.stderr:
            print(completed.stderr)

    if check and completed.returncode != 0:
        raise RuntimeError(f'Command failed ({completed.returncode}): {cmd}')
    return completed


print('Repo dir:', repo_dir)
print('UV project:', uv_project)
print('UV venv path:', os.environ['UV_PROJECT_ENVIRONMENT'])

# Ensure system Python has pip and uv
run('python -m ensurepip --upgrade', cwd=repo_dir, check=False, stream=True)
run('python -m pip install -U pip wheel setuptools', cwd=repo_dir, check=False, stream=True)
run('python -m pip install -U uv', cwd=repo_dir, check=False, stream=True)

active_python = 'python'
active_pip_cmd = 'python -m pip'

sync_cmd = shell_join(['uv', 'sync', '--project', uv_project, '--extra', uv_extra])
sync_result = run(sync_cmd, cwd=repo_dir, check=False, stream=True)
print('uv sync return code:', sync_result.returncode)

spacy_model_wheel = (
    'https://github.com/explosion/spacy-models/releases/download/'
    'en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl'
)
spacy_install_cmd = 'python -m spacy download en_core_web_sm'

if sync_result.returncode == 0:
    uv_python = uv_project / '.venv' / 'bin' / 'python'
    run(str(uv_python) + ' -m ensurepip --upgrade', cwd=repo_dir, check=False, stream=True)
    os.environ['PATH'] = f"{uv_python.parent}:{original_path}"

    active_python = str(uv_python)
    active_pip_cmd = f'{active_python} -m pip'
    spacy_install_cmd = shell_join(['uv', 'pip', 'install', '--python', uv_python, spacy_model_wheel])

    print(f'uv sync complete: {uv_project}')
    print('Activated venv bin:', uv_python.parent)
else:
    print('\nuv sync failed. Falling back to pip requirements install...')

    requirements_path = repo_dir / 'requirements.txt'
    pytorch_index = 'https://download.pytorch.org/whl/cu121'
    install_cmd = shell_join([
        'python', '-m', 'pip', 'install', '--extra-index-url', pytorch_index, '-r', requirements_path
    ])
    fallback_result = run(install_cmd, cwd=repo_dir, check=False, stream=True)

    if fallback_result.returncode != 0:
        print('\nFull requirements install failed. Using Colab-torch-compatible filtered install...')
        filtered = []
        skip_prefixes = ('torch==', 'torchvision==', 'torchaudio==')
        for raw in requirements_path.read_text(encoding='utf-8').splitlines():
            line = raw.strip()
            if not line or line.startswith('#'):
                continue
            if any(line.startswith(prefix) for prefix in skip_prefixes):
                continue
            filtered.append(line)

        temp_req = repo_dir / 'requirements.colab.filtered.txt'
        temp_req.write_text('\n'.join(filtered) + '\n', encoding='utf-8')

        run(
            'python - <<"PY"\n'
            'import torch\n'
            'import torchvision\n'
            'import torchaudio\n'
            'print("torch", torch.__version__)\n'
            'print("torchvision", torchvision.__version__)\n'
            'print("torchaudio", torchaudio.__version__)\n'
            'PY',
            cwd=repo_dir,
            check=False,
            stream=True
        )
        run(shell_join(['python', '-m', 'pip', 'install', '-r', temp_req]), cwd=repo_dir, stream=True)

# Install spaCy model with the chosen environment
run(spacy_install_cmd, cwd=repo_dir, stream=True)

# Ensure core runtime packages exist in the active environment
runtime_packages = ['transformers', 'pyyaml', 'tqdm', 'sentencepiece', 'accelerate']
run(f"{active_pip_cmd} install -U " + ' '.join(runtime_packages), cwd=repo_dir, check=False, stream=True)

# Quick sanity checks for downstream demo (against active interpreter)
run(f"{active_python} -c \"import torch, transformers, yaml; print(torch.__version__)\"", cwd=repo_dir, stream=True)
print('Dependency installation step complete.')

In [ ]:
import torch

print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('CUDA is not available. Switch Colab runtime to GPU before running the demo.')

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm
import shutil

STRATEGY = 'development'  # one of: development, validation, production
PROJECT_ROOT = Path(REPO_DIR)
LOCAL_DATA_PATH = PROJECT_ROOT / 'data'

required_files = [
    DRIVE_DATA_ROOT / f'indexes/{STRATEGY}/faiss.index',
    DRIVE_DATA_ROOT / f'indexes/{STRATEGY}/metadata.pkl',
    DRIVE_DATA_ROOT / f'indexes/{STRATEGY}/bm25_index.pkl',
    DRIVE_DATA_ROOT / f'processed/wiki_chunks_{STRATEGY}.jsonl',
]

missing = []
for p in tqdm(required_files, desc='Checking Drive artifacts', unit='file'):
    if not p.exists():
        missing.append(str(p))

if missing:
    raise FileNotFoundError('Missing required artifacts in DRIVE_DATA_ROOT:\n' + '\n'.join(missing))

# Ensure local repo data path points to Drive data root via symlink
if LOCAL_DATA_PATH.exists() or LOCAL_DATA_PATH.is_symlink():
    if LOCAL_DATA_PATH.is_symlink():
        current_target = LOCAL_DATA_PATH.resolve()
        if current_target != DRIVE_DATA_ROOT.resolve():
            LOCAL_DATA_PATH.unlink()
            LOCAL_DATA_PATH.symlink_to(DRIVE_DATA_ROOT, target_is_directory=True)
            print(f'Updated data symlink: {LOCAL_DATA_PATH} -> {DRIVE_DATA_ROOT}')
        else:
            print(f'Data symlink already correct: {LOCAL_DATA_PATH} -> {current_target}')
    else:
        backup_path = PROJECT_ROOT / 'data.local_backup'
        if backup_path.exists():
            raise RuntimeError(
                f'Cannot replace local data directory because backup already exists: {backup_path}. Resolve manually.'
            )
        shutil.move(str(LOCAL_DATA_PATH), str(backup_path))
        LOCAL_DATA_PATH.symlink_to(DRIVE_DATA_ROOT, target_is_directory=True)
        print(f'Moved existing local data to: {backup_path}')
        print(f'Created data symlink: {LOCAL_DATA_PATH} -> {DRIVE_DATA_ROOT}')
else:
    LOCAL_DATA_PATH.symlink_to(DRIVE_DATA_ROOT, target_is_directory=True)
    print(f'Created data symlink: {LOCAL_DATA_PATH} -> {DRIVE_DATA_ROOT}')

print('Artifact preflight complete for strategy:', STRATEGY)

In [ ]:
import yaml
from pathlib import Path

BASE_CONFIG = Path(REPO_DIR) / 'config.yaml'
COLAB_CONFIG = Path(REPO_DIR) / 'config.colab.longt5.yaml'

with open(BASE_CONFIG, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['models']['generator'] = 'google/long-t5-tglobal-base'
cfg['processing']['device'] = 'cuda'
cfg['generation']['load_in_8bit'] = False

# Safety defaults for Colab stability (adjust as needed)
cfg['generation']['max_new_tokens'] = 256
cfg['retrieval']['top_k'] = 5

with open(COLAB_CONFIG, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print('Wrote runtime config:', COLAB_CONFIG)

In [ ]:
import sys
import json
import time
from datetime import datetime
from pathlib import Path
import numpy as np

sys.path.insert(0, str(REPO_DIR))

from src.pipelines import BaselineRAGPipeline

def make_json_serializable(obj):
    if isinstance(obj, dict):
        return {k: make_json_serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_json_serializable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj

pipeline = BaselineRAGPipeline.from_config(
    config_path=str(Path(REPO_DIR) / 'config.colab.longt5.yaml'),
    strategy=STRATEGY
)

sample_queries = [
    'What is artificial intelligence?',
    'How do machines learn from data?',
    'What is machine learning? How does it differ from traditional programming?'
]

all_results = []
for i, query in enumerate(sample_queries, start=1):
    t0 = time.time()
    result = pipeline.run(query, top_k=cfg['retrieval']['top_k'])
    elapsed = time.time() - t0

    all_results.append({
        'query_index': i,
        'query': query,
        'timestamp': datetime.now().isoformat(),
        'latency_sec': round(elapsed, 3),
        'result': make_json_serializable(result),
    })

print(f'Completed {len(all_results)} queries')

In [ ]:
for item in all_results:
    print('=' * 100)
    print(f"Query {item['query_index']}: {item['query']}")
    print('-' * 100)

    result = item['result']
    retrieval = result.get('retrieval_metadata', {})
    pairs = result.get('claim_evidence_pairs', [])

    print('Latency (sec):', item.get('latency_sec'))
    print('Retrieved chunks:', retrieval.get('num_retrieved'))
    print('Top score:', retrieval.get('top_score'))

    print('Draft response:')
    print(result.get('draft_response', ''))

    print('Claims extracted:', len(pairs))
    if pairs:
        first_pair = pairs[0]
        spans = first_pair.get('evidence_spans', [])
        print('Top evidence candidates (up to 2):')
        for span in spans[:2]:
            text = span.get('text', '')
            print(f"- {span.get('doc_id')}#{span.get('sent_id')}: {text[:180]}")

    print()

In [ ]:
import json
from datetime import datetime
from pathlib import Path
import shutil

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
outputs_dir = Path(REPO_DIR) / 'outputs'
outputs_dir.mkdir(parents=True, exist_ok=True)
output_file = outputs_dir / f'rag_demo_results_{ts}.json'

# Keep the same top-level shape as scripts/demo_baseline_rag.py
json_results = []
for item in all_results:
    json_results.append({
        'query_index': item['query_index'],
        'query': item['query'],
        'timestamp': item['timestamp'],
        'result': item['result'],
    })

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(json_results, f, ensure_ascii=False, indent=2)

print('Saved locally:', output_file)

# Optional lightweight copy to Drive
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/AIST-FYP/outputs')
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
drive_output_file = DRIVE_OUTPUT_DIR / output_file.name
shutil.copy2(output_file, drive_output_file)
print('Copied to Drive:', drive_output_file)

## Troubleshooting

- If you hit OOM, reduce `cfg['generation']['max_new_tokens']` to 128 and run fewer queries.
- If artifacts are missing, verify `DRIVE_DATA_ROOT` and selected `STRATEGY`.
- If model load fails, re-check GPU runtime and rerun setup cells.
- This notebook expects a local repo at `/content/AIST-FYP` and a Drive artifact tree at `DRIVE_DATA_ROOT`.
- If symlink creation fails, ensure `PROJECT_ROOT/data` is not locked and rerun the artifact cell.